In [1]:
!pip install -q langchain-openai langchain-core requests

In [2]:
%pip install -q -U langchain-groq

In [3]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [4]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2
)

In [5]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [6]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [7]:
multiply.name

'multiply'

In [8]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [9]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [10]:
llm.invoke('hi')

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "hi". We need to respond. The user hasn\'t asked a question. We should greet back. Probably a friendly response.'}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 72, 'total_tokens': 119, 'completion_time': 0.048815969, 'completion_tokens_details': {'reasoning_tokens': 29}, 'prompt_time': 0.003447821, 'prompt_tokens_details': None, 'queue_time': 0.170042935, 'total_time': 0.05226379}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_d3e146e1a5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16d07-c7cf-7721-8fcc-5969d52f6359-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 47, 'total_tokens': 119, 'output_token_details': {'reasoning': 29}})

In [11]:
llm_with_tools = llm.bind_tools([multiply])

In [12]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="Hello! I'm just a bunch of code, but I'm here and ready to help. How can I assist you today?", additional_kwargs={'reasoning_content': 'User says "Hi how are you". We respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 133, 'total_tokens': 179, 'completion_time': 0.048163896, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.007575533, 'prompt_tokens_details': None, 'queue_time': 0.030073468, 'total_time': 0.055739429}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_92d51d08e5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16d07-ca28-7024-973d-8400b52717a4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 133, 'output_tokens': 46, 'total_tokens': 179, 'output_token_details': {'reasoning': 13}})

In [13]:
query = HumanMessage('can you multiply 3 with 1000')

In [14]:
messages=[query]

In [15]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [16]:
results=llm_with_tools.invoke(messages)

In [17]:
messages.append(results)

In [18]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the multiply function.', 'tool_calls': [{'id': 'fc_797cae2a-2e07-42ae-9f12-f21f20b8ee0e', 'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 138, 'total_tokens': 174, 'completion_time': 0.040805325, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.009270834, 'prompt_tokens_details': None, 'queue_time': 0.366150097, 'total_time': 0.050076159}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_75c733514d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16d07-cb0f-7434-ad53-530bf3b90548-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'fc_797cae2a-2e07-42ae-9f12-f21f20b8ee

In [19]:
print(results.tool_calls)
tool_result = multiply.invoke(results.tool_calls[0]["args"])
print(tool_result)

[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'fc_797cae2a-2e07-42ae-9f12-f21f20b8ee0e', 'type': 'tool_call'}]
3000


In [20]:
llm_with_tools.invoke(messages).content

''

TOOL CREATION

In [21]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [22]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [23]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1792800001,
 'time_last_update_utc': 'Sat, 24 Oct 2026 00:00:01 +0000',
 'time_next_update_unix': 1792886401,
 'time_next_update_utc': 'Sun, 25 Oct 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.0365}

In [36]:
convert.invoke({'base_currency_value':12, 'conversion_rate':95.0365})

1140.438

In [37]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [38]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [39]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [40]:
ai_message = llm_with_tools.invoke(messages)

In [41]:
ai_message


AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the get_conversion_factor function. Then use convert function. The user asks: "What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd". So we need to call get_conversion_factor with base_currency: "INR", target_currency: "USD". Then we need to convert 10 INR to USD. The convert function expects base_currency_value: number. But we need to specify base_currency_value: 10. But we also need to know the conversion factor. The convert function likely uses the conversion factor from the previous call. But we need to call convert with base_currency_value: 10. The function signature: convert(_:{base_currency_value:number}) => any. So we call convert with base_currency_value: 10. That will return the converted value. Then we need to respond with the conversion factor and the converted amount. We\'ll call get_conversion_factor first. Then call convert. We\'ll produce the answe

In [42]:
messages.append(ai_message)

In [43]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'fc_11b63138-cb75-43b6-a18e-a15eb9c0cdd1',
  'type': 'tool_call'}]

In [44]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [45]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the get_conversion_factor function. Then use convert function. The user asks: "What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd". So we need to call get_conversion_factor with base_currency: "INR", target_currency: "USD". Then we need to convert 10 INR to USD. The convert function expects base_currency_value: number. But we need to specify base_currency_value: 10. But we also need to know the conversion factor. The convert function likely uses the conversion factor from the previous call. But we need to call convert with base_currency_value: 10. The function signature: convert(_:{base_currency_value:number}) => any. So we call convert with base_currency_value: 10. That will return the converte

In [46]:
llm_with_tools.invoke(messages).content

'**Conversion factor (INR → USD):** 1\u202fINR = **0.01052\u202fUSD**  \n*(based on the latest exchange‑rate data)*  \n\n**10\u202fINR in USD:**  \n\\(10 \\times 0.01052 = 0.1052\\)\u202fUSD  \n\nSo, 10\u202fINR is approximately **$0.11** (rounded to the nearest cent).'

In [35]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, ToolMessage

agent_executor = create_agent(
    model=llm,
    tools=[get_conversion_factor, convert],
    system_prompt="You are a helpful unit conversion assistant."
)

user_query = "Hi, how are you?"

print("> Entering agent execution...")

response = agent_executor.invoke({
    "messages": [HumanMessage(content=user_query)]
})

for message in response["messages"]:
    if getattr(message, "tool_calls", None):
        for tool_call in message.tool_calls:
            print(f"> Calling tool: {tool_call['name']}")
            print(f"> Arguments: {tool_call['args']}")
    elif isinstance(message, ToolMessage):
        print(f"> Tool result: {message.content}")

print(response["messages"][-1].content)
print("> Finished chain.")

> Entering agent execution...
Hello! I'm just a bunch of code, but I'm here and ready to help you. How can I assist you today?
> Finished chain.
